In [0]:
# Objetivo:
# Importar as bibliotecas utilizadas
# durante o processo de auditoria e
# construção da camada Silver.

# Justificativa:
# As bibliotecas são carregadas no
# início do notebook para ficarem
# disponíveis em todas as etapas do
# pipeline.

# Ação:
# Importa as bibliotecas necessárias
# para manipulação dos dados e acesso
# ao sistema de arquivos.

import json
import pandas as pd

from pathlib import Path

In [0]:
# Objetivo:
# Definir as configurações utilizadas
# durante a execução do notebook.

# Justificativa:
# Os caminhos são centralizados no
# config.json e na silver_metadata,
# garantindo consistência entre as
# camadas Bronze e Silver.

# Ação:
# Carrega a configuração oficial do
# projeto e filtra os metadados da
# entidade processada neste notebook.

CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

BASE_PATH = Path(
    config["environment"]["base_path"]
)

CONFIG_PATH = Path(
    config["paths"]["config_path"]
)

LOG_PATH = Path(
    config["paths"]["log_path"]
)

EXECUTION_DATE = (
    config["project"]["execution_date"]
)

SILVER_METADATA_PATH = (
    CONFIG_PATH
    / "silver_metadata"
)

df_silver_metadata = pd.read_parquet(
    SILVER_METADATA_PATH
)

metadata_dataset = (
    df_silver_metadata[
        df_silver_metadata["dataset"]
        == "municipios"
    ]
    .sort_values("ano")
    .reset_index(drop=True)
)

if set(metadata_dataset["ano"]) != {2023, 2024, 2025}:
    raise ValueError(
        "Metadados Silver incompletos "
        "para o dataset municipios."
    )

CAMINHO_BRONZE = Path(
    metadata_dataset.iloc[0][
        "bronze_path"
    ]
).parent

CAMINHO_SILVER = Path(
    metadata_dataset.iloc[0][
        "silver_path"
    ]
).parent

print(
    "CAMINHO_BRONZE:",
    CAMINHO_BRONZE
)

print(
    "CAMINHO_SILVER:",
    CAMINHO_SILVER
)

display(metadata_dataset)

# 1. Auditoria da Fonte de Dados - Base Municípios

> **Nota**
>
> Durante o desenvolvimento deste projeto foi utilizado o dicionário oficial
> dos Microdados da Avaliação da Alfabetização disponibilizado pelo INEP como
> referência para interpretação das variáveis, domínios e regras de negócio
> presentes nas bases de dados.

## 1.1 Leitura das Bases

**Contexto**

Os microdados da Avaliação da Alfabetização foram organizados na camada
Bronze por entidade e particionados por ano, preservando a estrutura dos
arquivos disponibilizados pelo INEP.

Nesta etapa são carregadas as bases referentes aos anos de 2023, 2024 e
2025 para realização da auditoria estrutural e preparação da camada Silver.

**Objetivo**

Carregar as bases da entidade Municípios referentes aos anos de 2023, 2024
e 2025 para análise da estrutura e padronização dos dados.

**Resultado esperado**

Obter três DataFrames correspondentes às bases de municípios dos anos de
2023, 2024 e 2025, prontos para as etapas de auditoria e transformação.

### Integração com o Silver Orquestrador

Este notebook não utiliza caminhos locais ou nomes de arquivos fixos.

Os caminhos da entidade `municipios` são obtidos da tabela:

```text
config/silver_metadata
```

A leitura é feita diretamente das partições Parquet da Bronze:

```text
bronze/municipios/ano=2023
bronze/municipios/ano=2024
bronze/municipios/ano=2025
```

A lógica de auditoria, limpeza e transformação construída originalmente permanece preservada.

In [0]:
# Objetivo:
# Carregar as partições anuais da
# entidade municipios na camada Bronze.

# Justificativa:
# A Bronze foi persistida em Parquet,
# organizada por entidade e ano.
# A Silver deve consumir diretamente
# essas partições governadas pelo
# silver_metadata.

# Ação:
# Lê as partições Bronze referentes
# aos anos de 2023, 2024 e 2025.

def caminho_bronze_ano(ano):
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    return Path(
        registro["bronze_path"]
    )


df_municipios_2023 = pd.read_parquet(
    caminho_bronze_ano(2023)
)

df_municipios_2024 = pd.read_parquet(
    caminho_bronze_ano(2024)
)

df_municipios_2025 = pd.read_parquet(
    caminho_bronze_ano(2025)
)

## 1.2 Inspeção Inicial da Estrutura

**Contexto**

Após o carregamento das bases da camada Bronze, realiza-se uma inspeção
inicial para compreender a estrutura dos dados disponibilizados pelo INEP
em cada ano da avaliação.

Essa verificação permite identificar a quantidade de registros, colunas,
tipos de dados e possíveis diferenças estruturais entre as bases antes do
processo de padronização da camada Silver.

**Objetivo**

Inspecionar a estrutura das bases de municípios dos anos de 2023, 2024 e
2025, identificando eventuais diferenças que possam impactar as etapas
seguintes do pipeline.

**Resultado esperado**

Obter uma visão inicial da estrutura das bases, permitindo identificar
alterações entre os anos e subsidiar as etapas de auditoria e padronização.

In [0]:
# Objetivo:
# Realizar uma inspeção inicial da
# estrutura das bases de municípios.

# Justificativa:
# A inspeção inicial permite verificar
# o esquema das bases e identificar
# possíveis alterações nas colunas
# disponibilizadas pelo INEP ao longo
# dos anos da avaliação.

# Ação:
# Exibe a estrutura das bases de
# municípios para comparação entre os
# anos de 2023, 2024 e 2025.

print(df_municipios_2023.columns.tolist())

print(df_municipios_2024.columns.tolist())

print(df_municipios_2025.columns.tolist())

## 1.3 Comparação dos Tipos de Dados

**Contexto**

Além da estrutura das colunas, é importante verificar se os tipos de dados
foram mantidos entre as bases de 2023, 2024 e 2025.

Diferenças de tipagem podem comprometer a padronização da camada Silver e
dificultar a integração das partições durante a construção da camada Gold.

**Objetivo**

Comparar os tipos de dados das bases de municípios dos anos de 2023, 2024
e 2025, identificando possíveis divergências estruturais.

**Resultado esperado**

Obter um relatório de compatibilidade dos tipos de dados, permitindo
identificar eventuais diferenças que necessitem de padronização antes da
persistência da camada Silver.

In [0]:
# Objetivo:
# Comparar os tipos de dados das
# bases de municípios dos anos de
# 2023, 2024 e 2025.

# Justificativa:
# Colunas com o mesmo nome podem
# apresentar tipos diferentes entre
# os anos ou novas variáveis podem
# ter sido incorporadas pelo INEP.

# Ação:
# Consolida os tipos de dados das
# três bases em um único relatório
# e classifica a compatibilidade
# entre os esquemas.

relatorio_dtypes = pd.DataFrame({
    "2023": df_municipios_2023.dtypes.astype(str),
    "2024": df_municipios_2024.dtypes.astype(str),
    "2025": df_municipios_2025.dtypes.astype(str)
})

relatorio_dtypes = relatorio_dtypes.replace("nan", pd.NA)


def classificar_status(linha):

    if (
        pd.notna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.isna(linha["2025"])
    ):
        return "Exclusiva de 2023"

    if (
        pd.isna(linha["2023"])
        and pd.notna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova a partir de 2024"

    if (
        pd.isna(linha["2023"])
        and pd.isna(linha["2024"])
        and pd.notna(linha["2025"])
    ):
        return "Nova em 2025"

    tipos = linha.dropna()

    if len(tipos.unique()) == 1:
        return "Compatível"

    return "Divergente"


relatorio_dtypes["status"] = relatorio_dtypes.apply(
    classificar_status,
    axis=1
)

display(relatorio_dtypes)

## 1.4 Análise dos Valores Ausentes

**Contexto**

Após a verificação da estrutura e dos tipos de dados, torna-se necessário
avaliar a presença de valores ausentes nas bases dos diferentes anos.

Essa análise permite identificar possíveis impactos sobre a qualidade dos
dados e definir se será necessário realizar algum tratamento antes da
persistência da camada Silver.

**Objetivo**

Identificar e quantificar a ocorrência de valores ausentes nas bases da
entidade Municípios, avaliando a necessidade de tratamento durante o
processo de transformação.

**Resultado esperado**

Obter um diagnóstico da completude dos dados, permitindo justificar as
decisões adotadas em relação ao tratamento de valores ausentes na camada
Silver.

In [0]:
# Objetivo:
# Analisar a ocorrência de valores
# ausentes nas bases da entidade
# Municípios.

# Justificativa:
# A identificação de valores ausentes
# permite avaliar a qualidade dos
# dados e definir eventuais ações de
# tratamento antes da persistência da
# camada Silver.

# Ação:
# Calcula a quantidade de valores
# ausentes por coluna para cada ano
# da avaliação.

for ano, df in [
    (2023, df_municipios_2023),
    (2024, df_municipios_2024),
    (2025, df_municipios_2025)
]:

    print(f"\nValores ausentes - {ano}")

    valores_ausentes = (
        df.isna()
          .sum()
          .loc[lambda s: s > 0]
          .sort_values(ascending=False)
          .to_frame("Valores Ausentes")
    )

    if valores_ausentes.empty:
        print("Nenhum valor ausente encontrado.")
    else:
        display(valores_ausentes)

### 1.4.1 Análise dos Resultados

**Contexto**

Após a identificação dos valores ausentes, torna-se necessário interpretar
os resultados obtidos para verificar se as ocorrências representam
inconsistências na base de origem ou se decorrem das características dos
dados disponibilizados pelo INEP.

**Objetivo**

Interpretar os resultados da análise de valores ausentes e justificar as
decisões adotadas durante a construção da camada Silver.

**Resultado esperado**

Documentar que as bases originais de municípios não apresentam valores
ausentes e registrar que as adaptações estruturais realizadas
posteriormente fazem parte do processo de padronização da camada Silver,
não caracterizando problemas de qualidade dos dados de origem.

## 1.5 Seleção das Colunas da Camada Silver

**Contexto**

A auditoria da estrutura das bases mostrou que o INEP ampliou a quantidade
de informações disponibilizadas a partir de 2024, incorporando colunas
relacionadas à distribuição dos estudantes por níveis de proficiência em
Língua Portuguesa.

Como essas informações possuem valor analítico para as etapas posteriores
do projeto, elas serão preservadas na camada Silver. Para garantir a
compatibilidade estrutural entre as partições anuais, as colunas ausentes
na base de 2023 serão incorporadas durante o processo de padronização.

Nesta etapa são selecionadas todas as colunas que comporão a Base
Municípios da camada Silver, assegurando que as bases dos anos de 2023,
2024 e 2025 compartilhem o mesmo esquema antes da persistência.

**Objetivo**

Definir o conjunto de atributos que comporá a Base Municípios da camada
Silver, preservando as informações relevantes para as análises e etapas
posteriores do pipeline.

**Resultado esperado**

Obter três bases com a mesma estrutura de colunas, aptas para a
persistência particionada da camada Silver e para posterior integração na
camada Gold.

In [0]:
# Objetivo:
# Selecionar as colunas que farão
# parte da Base Municípios da
# camada Silver.

# Justificativa:
# As bases de 2024 e 2025 passaram
# a disponibilizar colunas referentes
# aos níveis de proficiência em Língua
# Portuguesa. Essas informações possuem
# valor analítico e serão preservadas
# na camada Silver.

# Ação:
# Padroniza a estrutura das bases,
# criando as colunas ausentes na base
# de 2023 e selecionando o conjunto
# final de atributos da camada Silver.

colunas_silver = [
    "NU_ANO_AVALIACAO",
    "CO_UF",
    "SG_UF",
    "CO_MUNICIPIO",
    "NO_MUNICIPIO",
    "TP_SERIE",
    "ID_TIPO_REDE",
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP",
    "PC_ALUNO_NIVEL_0_LP",
    "PC_ALUNO_NIVEL_1_LP",
    "PC_ALUNO_NIVEL_2_LP",
    "PC_ALUNO_NIVEL_3_LP",
    "PC_ALUNO_NIVEL_4_LP",
    "PC_ALUNO_NIVEL_5_LP",
    "PC_ALUNO_NIVEL_6_LP",
    "PC_ALUNO_NIVEL_7_LP",
    "PC_ALUNO_NIVEL_8_LP"
]

# Adiciona à base de 2023 as colunas
# introduzidas pelo INEP a partir de
# 2024, preservando a compatibilidade
# estrutural entre as partições.

for coluna in colunas_silver:
    if coluna not in df_municipios_2023.columns:
        df_municipios_2023[coluna] = pd.NA

df_municipios_2023 = df_municipios_2023[colunas_silver].copy()
df_municipios_2024 = df_municipios_2024[colunas_silver].copy()
df_municipios_2025 = df_municipios_2025[colunas_silver].copy()

## 1.6 Validação da Estrutura da Camada Silver

**Contexto**

Após a seleção e padronização das colunas que comporão a Base Municípios da
camada Silver, torna-se necessário validar se as três bases apresentam a
mesma estrutura.

Essa verificação garante que todas as partições anuais compartilhem o mesmo
esquema, permitindo sua integração na camada Gold sem necessidade de
transformações estruturais adicionais.

**Objetivo**

Validar a estrutura das bases de municípios após a seleção e padronização
das colunas da camada Silver.

**Resultado esperado**

Obter três bases com a mesma quantidade de colunas e estrutura compatível,
prontas para a persistência particionada da camada Silver.

In [0]:
# Objetivo:
# Validar a estrutura das bases
# após a seleção e padronização
# das colunas da camada Silver.

# Justificativa:
# Todas as partições anuais devem
# possuir o mesmo esquema antes da
# persistência na camada Silver.

# Ação:
# Compara a quantidade de colunas
# das bases de 2023, 2024 e 2025.

validacao_colunas = pd.DataFrame({
    "Base": ["2023", "2024", "2025"],
    "Quantidade de Colunas": [
        len(df_municipios_2023.columns),
        len(df_municipios_2024.columns),
        len(df_municipios_2025.columns)
    ]
})

validacao_colunas["Status"] = (
    "Compatível"
    if validacao_colunas["Quantidade de Colunas"].nunique() == 1
    else "Divergente"
)

display(validacao_colunas)

## 1.7 Persistência da Base Municípios

**Contexto**

Após a auditoria estrutural, padronização das colunas e validação do
esquema, as bases da entidade Municípios estão preparadas para serem
persistidas na camada Silver.

Seguindo a arquitetura definida para este projeto, os dados são armazenados
de forma particionada por ano, preservando a organização do Data Lake e
facilitando futuras integrações na camada Gold.

**Objetivo**

Persistir as bases de municípios da camada Silver em partições anuais,
preservando a padronização estrutural obtida durante o processo de
transformação.

**Resultado esperado**

Obter três partições da Base Municípios na camada Silver,
correspondentes aos anos de 2023, 2024 e 2025, prontas para consumo
analítico e integração na camada Gold.

In [0]:
# Objetivo:
# Persistir as bases da entidade
# municipios na camada Silver.

# Justificativa:
# A persistência utiliza os caminhos
# e nomes de arquivo registrados na
# silver_metadata, eliminando caminhos
# fixos e mantendo o particionamento
# anual definido no setup.

# Ação:
# Grava as bases tratadas dos anos de
# 2023, 2024 e 2025 em CSV UTF-8.

for ano, df in [
    (2023, df_municipios_2023),
    (2024, df_municipios_2024),
    (2025, df_municipios_2025)
]:
    registro = metadata_dataset[
        metadata_dataset["ano"] == ano
    ].iloc[0]

    destino = Path(
        registro["silver_path"]
    )

    nome_arquivo = registro[
        "silver_file_name"
    ]

    destino.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        destino / nome_arquivo,
        sep=";",
        decimal=",",
        encoding="utf-8",
        index=False
    )

    print(
        f"Silver salva: "
        f"{destino / nome_arquivo}"
    )

# Conclusão

Ao longo deste notebook foi realizada a auditoria estrutural das bases da
entidade **Municípios** referentes aos anos de **2023**, **2024** e
**2025**, identificando a evolução do esquema de dados disponibilizado pelo
INEP.

A auditoria contemplou a verificação da estrutura das bases, a comparação
dos tipos de dados e a análise dos valores ausentes. Não foram
identificadas inconsistências relacionadas à completude dos dados nas bases
originais disponibilizadas pelo INEP, não sendo necessário aplicar
tratamentos de imputação ou exclusão de registros durante essa etapa.

A análise evidenciou que, a partir de 2024, foram incorporadas novas
colunas relacionadas à distribuição percentual dos estudantes por níveis de
proficiência em Língua Portuguesa. Considerando o potencial analítico
dessas informações, optou-se por preservá-las na camada Silver, realizando
a padronização do esquema por meio da inclusão dessas colunas na base de
2023.

Após a padronização das colunas e validação da estrutura, as bases foram
persistidas na camada **Silver**, mantendo a organização por entidade e o
particionamento por ano, conforme a arquitetura definida para o projeto.

Dessa forma, a Base Municípios da camada Silver passa a representar uma
versão padronizada, auditada e organizada dos dados consolidados por
município, constituindo uma fonte confiável para a construção dos
indicadores analíticos, das comparações entre metas e resultados e das
análises temporais desenvolvidas na camada Gold.